# Notebook 7. FP8

FP8 stores each weight as a sign, an exponent and a mantissa, like FP16 with fewer bits of each. Its grid is neither uniform nor a fixed codebook: the spacing between representable values doubles with every power of two, so small values are stored finely and large ones coarsely. This notebook shows that grid, explains why weights are scaled before the cast, compares the two FP8 formats, states what Pare's weight-only FP8 does and does not provide, and runs both formats on GPT-2.

Prerequisites are Notebooks 1, 2 and 6.

In [ ]:
import sys

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, "..")   # import Pare from this checkout; `pip install pare-quant` also works
torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Two formats

PyTorch provides two 8-bit floating-point dtypes.

| dtype | Exponent bits | Mantissa bits | Largest finite value | Spacing just below 1 |
|---|---|---|---|---|
| `torch.float8_e4m3fn` | 4 | 3 | 448 | 1/16 |
| `torch.float8_e5m2` | 5 | 2 | 57344 | 1/8 |

Not all 256 bit patterns are finite numbers: E4M3 reserves two for NaN and has 253 distinct finite values, and E5M2 reserves six for NaN and two for infinities and has 247.

E4M3 gives up range for one more bit of precision; E5M2 is the opposite trade, with the exponent range of FP16. The cell plots every representable E4M3 value in $[0, 4)$.

In [ ]:
from pare.core.functional import _FP8_E4M3_MAX, _FP8_E5M2_MAX

print(f"largest finite value: E4M3 {_FP8_E4M3_MAX}, E5M2 {_FP8_E5M2_MAX}")

values = torch.arange(0, 4, 1e-4).to(torch.float8_e4m3fn).float().unique()
values = values[values < 4.0]
fig, ax = plt.subplots(figsize=(9, 2))
ax.scatter(values, [0] * len(values), s=12)
ax.set_yticks([])
ax.set_xlabel("value")
ax.set_title(f"the {len(values)} E4M3 values in [0, 4): spacing doubles at each power of two")
plt.tight_layout()
plt.show()

## 2. Scale before casting

Weights have magnitudes of a few hundredths, far below the top of either format's range, and a plain cast would waste the large exponents. Pare's `quantize_fp8` divides each row by `absmax / fp8_max`, so that the largest value in the row lands exactly on the format's largest finite value, casts, and stores the per-row scale in FP32. This is the same per-row scale as RTN INT8; only the target grid differs. The cell is Pare's implementation and checks it against the library.

In [ ]:
from pare.core.dtype import QuantDtype
from pare.core.functional import dequantize_fp8 as pare_dequantize_fp8
from pare.core.functional import quantize_fp8 as pare_quantize_fp8

FP8 = {QuantDtype.FP8_E4M3: (torch.float8_e4m3fn, _FP8_E4M3_MAX),
       QuantDtype.FP8_E5M2: (torch.float8_e5m2, _FP8_E5M2_MAX)}


def fp8_scale(W, dtype):
    return W.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / FP8[dtype][1]


def quantize_fp8(W, scale, dtype):
    return (W / scale.clamp(min=1e-8)).to(FP8[dtype][0])


def dequantize_fp8(W_fp8, scale):
    return W_fp8.float() * scale


torch.manual_seed(0)
W = torch.randn(64, 256) * 0.05
for dtype in FP8:
    scale = fp8_scale(W, dtype)
    W_fp8 = quantize_fp8(W, scale, dtype)
    W_fp8_pare = pare_quantize_fp8(W, scale, dtype)
    assert torch.equal(W_fp8.view(torch.uint8), W_fp8_pare.view(torch.uint8))
    assert torch.allclose(dequantize_fp8(W_fp8, scale), pare_dequantize_fp8(W_fp8, scale))
    print(f"{dtype.name}: stored dtype {W_fp8.dtype}, reconstruction MAE "
          f"{(dequantize_fp8(W_fp8, scale) - W).abs().mean():.6f}")
print("quantize_fp8 and dequantize_fp8 match pare.core.functional")

## 3. E4M3 against E5M2

For Gaussian weights with no extreme values, the extra mantissa bit of E4M3 should matter and the extra range of E5M2 should not. The cell measures both, with asymmetric INT8 at the same per-row scale for reference, then puts one large value into a row and measures the error of that row's other 127 weights.

In [ ]:
def mse(a, b):
    return (a - b).pow(2).mean().item()


def int8_rows(W):
    # Asymmetric INT8 with one scale per row (Notebook 2).
    w_min = W.amin(dim=1, keepdim=True).clamp(max=0.0)
    w_max = W.amax(dim=1, keepdim=True).clamp(min=0.0)
    s = ((w_max - w_min) / 255.0).clamp(min=1e-8)
    z = torch.round(-128 - w_min / s).clamp(-128, 127)
    return s * (torch.clamp(torch.round(W / s + z), -128, 127) - z)


def fp8_rows(W, dtype):
    scale = fp8_scale(W, dtype)
    return dequantize_fp8(quantize_fp8(W, scale, dtype), scale)


torch.manual_seed(42)
W = torch.randn(64, 128)
W_out = W.clone()
W_out[0, 0] = 50.0                    # one large value in row 0

print("format      MSE, Gaussian rows    row 0 MSE without outlier    row 0 MSE with outlier")
for name, fn in [("INT8", int8_rows),
                 ("FP8 E4M3", lambda W: fp8_rows(W, QuantDtype.FP8_E4M3)),
                 ("FP8 E5M2", lambda W: fp8_rows(W, QuantDtype.FP8_E5M2))]:
    print(f"{name:10} {mse(W, fn(W)):18.3e} {mse(W[0, 1:], fn(W)[0, 1:]):26.3e} "
          f"{mse(W_out[0, 1:], fn(W_out)[0, 1:]):22.3e}")

On clean rows INT8 is far ahead of both FP8 formats, and E4M3 is ahead of E5M2 by a factor of about four. The outlier reverses one of these. INT8's step is set by the row maximum, so one value of 50 in a row of unit Gaussians multiplies the error of the other 127 weights by about a hundred. The FP8 formats store each value with a roughly constant relative precision, about $2^{-4}$ for E4M3, so the per-row scale only has to keep the row inside the finite range, and the other weights are almost unaffected. Among 8-bit weight formats, INT8 is the choice for rows without extreme values and a floating-point grid is the robust one when there are.

## 4. Weight-only FP8

Two different things are called FP8 inference.

1. **Weight-only, W8A16.** Weights are stored in FP8 and converted to FP16 or BF16 before the matmul, which runs as it does for FP16. This halves weight memory, needs only the FP8 storage dtypes, which PyTorch provides on CPU and on every GPU, and gives no speed gain from the matmul. This is what Pare implements.
2. **FP8 compute, W8A8.** Weights and activations stay in FP8 through the matmul on FP8 tensor cores. This needs Hopper (H100), Ada (RTX 40, L4, L40) or later GPUs; Ampere (A100, A40) has none. Pare does not implement it.

Memory is the whole benefit of Pare's FP8, and for that purpose it competes with INT8 at the same storage cost.

## 5. FP8 in Pare on GPT-2

`QuantConfig(dtype=QuantDtype.FP8_E4M3)` or `FP8_E5M2` with `scheme="rtn"` applies the per-row cast above to every linear layer. The cell compares the two formats and RTN INT8 against FP32 on GPT-2, with 20 evaluation windows of 512 tokens, on CPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from pare import QuantConfig, quantize
from pare.eval.perplexity import evaluate_perplexity

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)


def fresh_model():
    return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)


def ppl(model):
    return evaluate_perplexity(
        model, tokenizer, dataset="wikitext2", seq_len=512, n_samples=20, device="cpu"
    )


results = {"FP32": ppl(fresh_model())}
configs = {
    "RTN INT8 per row": QuantConfig(bits=8, scheme="rtn", granularity="per_channel"),
    "FP8 E4M3": QuantConfig(bits=8, dtype=QuantDtype.FP8_E4M3, scheme="rtn"),
    "FP8 E5M2": QuantConfig(bits=8, dtype=QuantDtype.FP8_E5M2, scheme="rtn"),
}
for label, cfg in configs.items():
    model = fresh_model()
    quantize(model, cfg)
    results[label] = ppl(model)

for label, value in results.items():
    print(f"{label:>18}: PPL {value:6.2f}  ({value - results['FP32']:+.2f})")

The ordering should follow Section 3: INT8 closest to FP32, then E4M3, then E5M2 with a visibly larger change. All three store eight bits per weight, so the differences are the price of the grid alone.

## Exercises

1. Plot the representable E5M2 values in $[0, 4)$ next to the E4M3 ones from Section 1 and count them. Why does E5M2 have fewer values in that interval?
2. In Section 3, scale the Gaussian weights by 1000 before quantizing. Does the ratio of the two errors change, and what does that tell you about the role of the per-row scale?
3. Replace the per-row scale with one scale for the whole matrix and repeat Section 5 for E4M3. How much does the per-row scale buy?

## Summary

| | INT8 | FP8 E4M3 | FP8 E5M2 |
|---|---|---|---|
| Distinct finite values | 256, equally spaced | 253, spacing doubles at each power of two | 247, coarser spacing, wider range |
| Largest finite value after scaling | 127 | 448 | 57344 |
| Scale in Pare | per row, FP32 | per row, FP32 | per row, FP32 |
| Hardware for Pare's weight-only path | any | any PyTorch device | any PyTorch device |
| Error on a row with one large value | grows with the value | almost unchanged | almost unchanged |

This completes the weight formats. The notebooks README summarizes which of Pare's schemes run where.